# 02 — Preprocessing (REVISI)

**Yang dipertahankan dari skripsi:** duplikat exact 21 dibuang / logis 139 dipertahankan, outlier IQR dideteksi tapi dipertahankan, imputasi hierarkis, rumus depresiasi domain-informed.

**Yang DIPERBAIKI di revisi ini:**
1. Path relatif (`../data/raw/...`), bukan `/Skripsi/...` atau `D:/Skripsi/...`.
2. `REFERENCE_YEAR = 2025` (data triwulan 2025), bukan hardcode 2026 — umur barang 2025 jadi 0, bukan 1. Untuk data baru di aplikasi, umur dihitung dinamis dari tahun berjalan.
3. **Imputasi TIDAK lagi dihitung di full data lalu dipakai begitu saja.** Notebook ini menyimpan DUA file:
   - `inventaris_processed.csv` — versi ter-imputasi, untuk EDA/dashboard (praktis).
   - `inventaris_for_model.csv` — versi `harga_per_unit`/`depresiasi_aset` masih NaN, untuk training di notebook 03 di mana imputasi dihitung HANYA dari train tiap fold (tanpa leakage).
4. `preprocessing_reference.pkl` diperbaiki: menyimpan mean `harga_per_unit` (bukan `harga` total), median, dan masa manfaat.
5. Target biner dibuat di sini, tapi `kondisi_asli` tetap disimpan untuk audit. Metadata `nama_barang`, `nama_ruangan`, `triwulan` tetap disimpan (bukan fitur model, tapi perlu untuk dashboard).

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path

pd.options.display.float_format = '{:.2f}'.format
REFERENCE_YEAR = 2025  # data triwulan 2025; JANGAN 2026
RAW = Path('../data/raw/inventaris_raw.csv')
PROC = Path('../data/processed')
PROC.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(RAW)
print('awal:', len(df))

awal: 2322


In [2]:
# --- Cleaning: strip + buang duplikat EXACT saja ---
for col in ['bahan','kategori','kondisi','nama_ruangan']:
    df[col] = df[col].astype(str).str.strip()
print('duplikat exact:', int(df.duplicated().sum()))
subset = ['triwulan','nama_ruangan','nama_barang','tahun_pembelian']
print('duplikat logis (subset di atas, DIPERTAHANKAN karena batch stok beda):', int(df.duplicated(subset=subset).sum()))
df = df.drop_duplicates().copy()
print('setelah cleaning:', len(df), '(harus 2301)')

duplikat exact: 21
duplikat logis (subset di atas, DIPERTAHANKAN karena batch stok beda): 139
setelah cleaning: 2301 (harus 2301)


In [3]:
# --- Outlier IQR: deteksi saja, TIDAK dibuang ---
for col in ['tahun_pembelian','jumlah','harga']:
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    iqr = q3-q1
    n = int(((df[col] < q1-1.5*iqr) | (df[col] > q3+1.5*iqr)).sum())
    print(f'{col}: {n} outlier ({n/len(df)*100:.2f}%) — dipertahankan (kondisi nyata)')

tahun_pembelian: 0 outlier (0.00%) — dipertahankan (kondisi nyata)
jumlah: 261 outlier (11.34%) — dipertahankan (kondisi nyata)
harga: 145 outlier (6.30%) — dipertahankan (kondisi nyata)


In [4]:
# --- Feature engineering ---
df['umur'] = REFERENCE_YEAR - df['tahun_pembelian']
df['harga_per_unit'] = df['harga'] / df['jumlah']
print('missing harga_per_unit sebelum imputasi:', int(df['harga_per_unit'].isna().sum()), '(harus ~1289)')
print(df[['umur','harga_per_unit']].describe().T)

missing harga_per_unit sebelum imputasi: 1289 (harus ~1289)
                 count       mean        std     min       25%        50%  \
umur           2301.00       6.08       3.62    0.00      3.00       6.00   
harga_per_unit 1012.00 4070705.35 6310597.30 4000.00 555000.00 1850031.25   

                      75%         max  
umur                 9.00       14.00  
harga_per_unit 5400000.00 57002629.00  


In [5]:
def depresiasi_row(row):
    umur, harga = row['umur'], row['harga_per_unit']
    if pd.isna(harga):
        return np.nan
    if row['kategori'] == 'Elektronik':
        nilai_buku = harga
        for _ in range(int(umur)):
            nilai_buku -= nilai_buku * (2/5)
        dep = harga - nilai_buku
    else:
        dep = (harga/10) * umur
    return round(min(dep, harga), 2)

df['depresiasi_aset'] = df.apply(depresiasi_row, axis=1)
print('depresiasi: min/median/max =', df['depresiasi_aset'].min(), df['depresiasi_aset'].median(), df['depresiasi_aset'].max())

depresiasi: min/median/max = 0.0 1416383.57 44690061.14


In [6]:
# --- Target biner + simpan kondisi asli ---
df['kondisi_asli'] = df['kondisi']
df['kondisi_biner'] = df['kondisi'].map({'Baik': 0, 'Kurang Baik': 1, 'Rusak Berat': 1})
print(df['kondisi_biner'].value_counts().to_dict())
assert df['kondisi_biner'].isna().sum() == 0

{0: 1889, 1: 412}


In [7]:
# --- Simpan VERSI UNTUK MODEL (NaN dipertahankan, imputasi nanti di dalam fold notebook 03) ---
cols_model = ['triwulan','nama_ruangan','nama_barang','bahan','kategori','umur','harga_per_unit','depresiasi_aset','kondisi_asli','kondisi_biner']
df_model = df[cols_model].copy()
df_model.to_csv(PROC/'inventaris_for_model.csv', index=False)
print('saved inventaris_for_model.csv:', df_model.shape, '| NaN harga_per_unit:', int(df_model['harga_per_unit'].isna().sum()))

saved inventaris_for_model.csv: (2301, 10) | NaN harga_per_unit: 1289


In [8]:
# --- Imputasi hierarkis HANYA untuk file EDA/dashboard (dan untuk referensi nilai) ---
# Urutan: mean per nama_barang -> mean per bahan+kategori -> median. Nilai-nilai ini
# disimpan ke reference.pkl dan DIPAKAI KEMBALI di notebook 03 / FastAPI untuk data baru.
df_imp = df.copy()
df_imp['harga_per_unit'] = df_imp.groupby('nama_barang')['harga_per_unit'].transform(lambda x: x.fillna(x.mean()))
df_imp['harga_per_unit'] = df_imp.groupby(['bahan','kategori'])['harga_per_unit'].transform(lambda x: x.fillna(x.mean()))
df_imp['harga_per_unit'] = df_imp['harga_per_unit'].fillna(df_imp['harga_per_unit'].median())
df_imp['depresiasi_aset'] = df_imp.apply(depresiasi_row, axis=1)
print('missing setelah imputasi (versi EDA):', int(df_imp['harga_per_unit'].isna().sum()))
cols_proc = ['bahan','kategori','umur','harga_per_unit','depresiasi_aset','kondisi_asli','kondisi_biner']
df_imp[cols_proc].to_csv(PROC/'inventaris_processed.csv', index=False)
print('saved inventaris_processed.csv:', df_imp[cols_proc].shape)

missing setelah imputasi (versi EDA): 0
saved inventaris_processed.csv: (2301, 7)


In [9]:
# --- Reference v2 (DIPERBAIKI: pakai harga_per_unit, bukan harga total) ---
reference = {
    'reference_year': REFERENCE_YEAR,
    'median_harga_per_unit': float(df['harga_per_unit'].median()),
    'harga_per_unit_nama_mean': df.groupby('nama_barang')['harga_per_unit'].mean().to_dict(),
    'harga_per_unit_group_mean': df.groupby(['bahan','kategori'])['harga_per_unit'].mean().to_dict(),
    'bahan_mapping': df.groupby('nama_barang')['bahan'].agg(lambda x: x.mode().iloc[0]).to_dict(),
    'kategori_mapping': df.groupby('nama_barang')['kategori'].agg(lambda x: x.mode().iloc[0]).to_dict(),
    'masa_manfaat': {'Elektronik': 5, 'Furnitur': 10, 'Pelengkap': 10},
    'depresiasi_method': {'Elektronik': 'double_declining', 'Furnitur': 'straight_line', 'Pelengkap': 'straight_line'},
}
joblib.dump(reference, '../models/preprocessing_reference_v2.pkl')
print('saved preprocessing_reference_v2.pkl')
print('statistik akhir (versi ter-imputasi):')
display(df_imp[['umur','harga_per_unit','depresiasi_aset']].describe().T)

saved preprocessing_reference_v2.pkl
statistik akhir (versi ter-imputasi):


,count,mean,std,min,25%,50%,75%,max
umur,2301.00,6.08,3.62,0.00,3.00,6.00,9.00,14.00
harga_per_unit,2301.00,3948410.25,4791405.15,4000.00,1412272.73,3135000.00,4846432.59,57002629.00
depresiasi_aset,2301.00,2652695.73,3367670.60,0.00,461709.77,1575666.96,3821086.73,44690061.14


## Kesimpulan 02

- File model: `data/processed/inventaris_for_model.csv` (2.301 baris, NaN dipertahankan) → dipakai notebook 03.
- File EDA: `data/processed/inventaris_processed.csv` (ter-imputasi) → dipakai dashboard.
- Jangan pernah pakai file EDA untuk klaim evaluasi tanpa leakage — evaluasi yang benar ada di notebook 03, di mana imputasi/scaler/encoder/SMOTE-NC semuanya di-fit HANYA di train tiap fold.

Lanjut ke `03_xgboost_smotenc_shap.ipynb`.